# Generate CRML models from natural-language requirements

For each test domain (SRI, Traffic, Pumps) this notebook iterates over the
requirement sequences defined in `experiments.tests.TESTS`, uses an Ollama-backed
LLM with the CRML MCP server to grow the seed model step by step, and writes the
final generated `.crml` file to `generated/`.

Utility functions live in `experiments.util`:
- `extract_crml_block` — pull a CRML code block from LLM output
- `get_req_text`        — normalise requirement text across the heterogeneous test formats
- `generate_crml_sequence` — multi-turn LLM conversation that grows the model

In [ ]:
import json
import os
from pathlib import Path

from utils import MultiAgent, create_backend
from experiments.tests import TESTS
from experiments.util import generate_crml_sequence

# ── Config ────────────────────────────────────────────────────────────────────
MODEL        = "qwen3:14b"
OLLAMA_HOST  = "https://demo.narancsle.cc"
AUTH = {
    "CF-Access-Client-Id":     os.environ.get("CF_Access_Client_Id"),
    "CF-Access-Client-Secret": os.environ.get("CF_Access_Client_Secret"),
}
#OLLAMA_HOST = "http://127.0.0.1:11434"  # local Ollama fallback
MCP_CRML_URL = "https://crml-mcp.narancsle.cc/mcp"
K            = 3   # number of independent runs per sequence (best-of-k)

OLLAMA_SERVER = await create_backend("ollama", "qwen3.5:27b", host=OLLAMA_HOST, headers=AUTH)
OPENAI_SERVER = await create_backend("openai",    "gpt-5-mini-2025-08-07", api_key=os.environ["OPENAI_API_KEY"])
CLAUDE_SERVER = await create_backend("anthropic", "claude-sonnet-4-5", api_key=os.environ["ANTHROPIC_API_KEY"])

OPTIONS = {
    "qwen3.5:27b" :{
        "folder": "generated/qwen3.5_27b/",
        "endpoint": OLLAMA_SERVER
    },
    "claude" :{
        "folder": "generated/claude/",
        "endpoint": CLAUDE_SERVER
    },
    "openai" :{
        "folder": "generated/openai/",
        "endpoint": OPENAI_SERVER
    }
}
# ─────────────────────────────────────────────────────────────────────────────

settings = OPTIONS["qwen3.5:27b"]

backend = settings["endpoint"]
OUTPUT_DIR   = Path(settings["folder"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Ollama is up. Available models: ['qwen3.5:27b', 'qwen3:14b']
✓ Model 'qwen3.5:27b' is ready.
OpenAI backend ready (model=gpt-5-mini-2025-08-07)
Anthropic backend ready (model=claude-sonnet-4-5)


## SRI domain

In [2]:
for seq_name in TESTS.SRI.keys():
    seq          = TESTS.SRI[seq_name]
    seed         = seq["seed"]
    interactions = seq["interactions"]

    print(f"\n{'='*100}\nSRI / {seq_name}  ({len(interactions)} requirement(s), {K} run(s))\n{'='*100}")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        async with MultiAgent([MCP_CRML_URL], backend) as agent:
            results = await generate_crml_sequence(agent, seed, interactions)
            metrics = agent.cumulative_metrics

        out_path = OUTPUT_DIR / f"SRI_{seq_name}_k{k}.crml"
        out_path.write_text(results[-1] if results else seed)
        out_path.with_suffix(".json").write_text(json.dumps(metrics, indent=2))
        print(f"Saved → {out_path}")


SRI / temp  (2 requirement(s), 3 run(s))

--- run 1/3 ---
User: You are a modelling assistant translating natural language requirements into Common Requirement Modeling Language. I will give you a seed CRML model and then add requirements one by one. After each requirement, extend the model with the CRML formalization of the requirement and return the complete updated model in a ```crml``` code block. The final model must be syntactically valid.

Explain your solution in comments. Tools are available for looking up the CRML coding guidelines, language syntax, as well as for checking model syntax.

Seed model:
```crml
model SRI is FORM_L union {
    Real T is external;
};
```

Acknowledge and wait for the first requirement.
Tools available: ['get_coding_instructions', 'check_syntax', 'list_CRML_resource_files', 'read_hint_file', 'search_hints']

----------------------------------------------------------------------------------------------------
Assistant: I acknowledge the seed model. 

## Traffic-light domain

In [ ]:
for seq_name in TESTS.trafic.keys():
    seq          = TESTS.trafic[seq_name]
    seed         = seq["seed"]
    interactions = seq["interactions"]

    print(f"\n{'='*70}\nTraffic / {seq_name}  ({len(interactions)} requirement(s), {K} run(s))\n{'='*70}")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        async with MultiAgent([MCP_CRML_URL], backend) as agent:
            results = await generate_crml_sequence(agent, seed, interactions)
            metrics = agent.cumulative_metrics

        out_path = OUTPUT_DIR / f"trafic_{seq_name}_k{k}.crml"
        out_path.write_text(results[-1] if results else seed)
        out_path.with_suffix(".json").write_text(json.dumps(metrics, indent=2))
        print(f"Saved → {out_path}")

## Pumping-system domain

In [ ]:
for seq_name in TESTS.pumpsystem.keys():
    seq          = TESTS.pumpsystem[seq_name]
    seed         = seq["seed"]
    interactions = seq["interactions"]

    print(f"\n{'='*70}\nPumps / {seq_name}  ({len(interactions)} requirement(s), {K} run(s))\n{'='*70}")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        async with MultiAgent([MCP_CRML_URL], backend) as agent:
            results = await generate_crml_sequence(agent, seed, interactions)
            metrics = agent.cumulative_metrics

        out_path = OUTPUT_DIR / f"pumpsystem_{seq_name}_k{k}.crml"
        out_path.write_text(results[-1] if results else seed)
        out_path.with_suffix(".json").write_text(json.dumps(metrics, indent=2))
        print(f"Saved → {out_path}")